In [ ]:
!pip install catboost xgboost lightgbm optuna shap imbalanced-learn


In [ ]:
import os
import sys
import pathlib
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Windows CP949 인코딩 문제 방지
if sys.platform == 'win32':
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass

from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)
from imblearn.over_sampling import SMOTE

import lightgbm as lgb
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

# xai 패키지 경로 추가
try:
    import google.colab
    sys.path.insert(0, '/content/drive/MyDrive/GoogleAI_contest/Taehyun')
except ImportError:
    current_dir = pathlib.Path(os.getcwd())
    sys.path.insert(0, str(current_dir))
try:
    from xai import ShapAnalyzer
except ImportError:
    from xai.analyzer import ShapAnalyzer

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False


# 1. 글로벌 경로 및 설정


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = pathlib.Path('/content/drive/MyDrive/GoogleAI_contest/Taehyun')
except ImportError:
    BASE_DIR = pathlib.Path(r"c:\ML4")
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "tabular"
PATIENT_PATH = PROCESSED_DIR / "patient_level_all_v2.csv"
PLOT_DIR = BASE_DIR / "report" / "plots"
os.makedirs(PLOT_DIR, exist_ok=True)

TARGET_COL = "label"  # 0: CN (Normal, 111명), 1: Abnormal (MCI+Dem, 63명)
DROP_COLS = ["EMAIL", "date", "DIAG_NM", "original_label", TARGET_COL, "fold"]

RANDOM_STATE = 42
N_SPLITS = 5
FORWARD_SELECTION_MAX_FEATURES = 40


# 2. 데이터 전처리 (Data Preprocessing - Load, Impute, Engineering)


### 📌 [참고] 원본 데이터 전처리 및 patient_level_all_v2.csv 생성 로직
이 부분은 Raw 시계열 데이터에서 mean, std 등의 파생 변수를 추출하여 현재 노트북에서 사용하는 CSV 파일을 만들어낸 원본 스크립트입니다.

In [ ]:
# (참고) 원시 시계열 데이터를 환자 수준으로 사전 집계하는 기초 전처리 코드입니다.
# 파일 경로 설정이 다를 수 있으므로 코드 구조만 참고하시기 바랍니다.

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold


BASE_DIR = Path(__file__).resolve().parent
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
OUT_DIR = DATA_DIR / "processed" / "tabular"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_SPLITS = 5


def find_csv(filename: str) -> Path:
    """Find a CSV in data/ first, then data/raw/ including .part0 copies."""
    candidates = [
        DATA_DIR / filename,
        RAW_DIR / filename,
        RAW_DIR / f"{filename}.part0",
    ]
    for path in candidates:
        if path.exists():
            return path

    matches = sorted(DATA_DIR.rglob(filename)) + sorted(DATA_DIR.rglob(f"{filename}.part0"))
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Cannot find {filename} under {DATA_DIR}")


def read_csv_flexible(path: Path) -> pd.DataFrame:
    for encoding in ("utf-8", "utf-8-sig", "cp949", "euc-kr"):
        try:
            return pd.read_csv(path, encoding=encoding, dtype=str, low_memory=False)
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f"Failed to read {path} with common Korean/UTF-8 encodings.")


def coerce_numeric_columns(df: pd.DataFrame, keep: set[str]) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if col not in keep:
            converted = pd.to_numeric(df[col], errors="coerce")
            if not converted.isna().all():
                df[col] = converted
    return df


def preprocess_label(label_df: pd.DataFrame) -> pd.DataFrame:
    label_df = label_df.copy()
    if "SAMPLE_EMAIL" in label_df.columns:
        label_df = label_df.rename(columns={"SAMPLE_EMAIL": "EMAIL"})

    required = {"EMAIL", "DIAG_NM"}
    missing = required - set(label_df.columns)
    if missing:
        raise ValueError(f"Label file is missing columns: {sorted(missing)}")

    original_label_map = {"CN": 0, "MCI": 1, "Dem": 2, "Dementia": 2}
    binary_label_map = {"CN": 0, "MCI": 1, "Dem": 1, "Dementia": 1}
    label_df["original_label"] = label_df["DIAG_NM"].map(original_label_map)
    label_df["label"] = label_df["DIAG_NM"].map(binary_label_map)

    if label_df["label"].isna().any():
        counts = label_df["DIAG_NM"].value_counts(dropna=False)
        raise ValueError(f"Unmapped DIAG_NM values found:\n{counts}")

    return label_df[["EMAIL", "DIAG_NM", "original_label", "label"]].drop_duplicates("EMAIL")


def parse_slash_sequence(value, dtype=float) -> np.ndarray:
    if pd.isna(value):
        return np.array([], dtype=float)

    text = str(value).strip()
    if not text or text == "...":
        return np.array([], dtype=float)

    values = []
    for token in text.split("/"):
        token = token.strip()
        if not token or token == "...":
            continue
        try:
            values.append(dtype(token))
        except ValueError:
            continue

    return np.array(values, dtype=float)


def numeric_sequence_stats(seq: np.ndarray, prefix: str, remove_zero: bool = False) -> dict[str, float]:
    arr = np.asarray(seq, dtype=float)
    arr = arr[arr != -1]
    if remove_zero:
        arr = arr[arr != 0]

    if len(arr) == 0:
        return {
            f"{prefix}_mean": np.nan,
            f"{prefix}_std": np.nan,
            f"{prefix}_var": np.nan,
            f"{prefix}_min": np.nan,
            f"{prefix}_max": np.nan,
            f"{prefix}_median": np.nan,
            f"{prefix}_q25": np.nan,
            f"{prefix}_q75": np.nan,
            f"{prefix}_iqr": np.nan,
            f"{prefix}_valid_count": 0,
        }

    q25 = np.percentile(arr, 25)
    q75 = np.percentile(arr, 75)
    return {
        f"{prefix}_mean": float(np.mean(arr)),
        f"{prefix}_std": float(np.std(arr)),
        f"{prefix}_var": float(np.var(arr)),
        f"{prefix}_min": float(np.min(arr)),
        f"{prefix}_max": float(np.max(arr)),
        f"{prefix}_median": float(np.median(arr)),
        f"{prefix}_q25": float(q25),
        f"{prefix}_q75": float(q75),
        f"{prefix}_iqr": float(q75 - q25),
        f"{prefix}_valid_count": int(len(arr)),
    }


def activity_class_features(seq: np.ndarray) -> dict[str, float]:
    arr = np.asarray(seq, dtype=float)
    arr = arr[arr != -1]
    total = len(arr)
    out: dict[str, float] = {}

    for level in range(6):
        count = int(np.sum(arr == level))
        out[f"activity_class_{level}_count"] = count
        out[f"activity_class_{level}_ratio"] = count / total if total else np.nan

    out["activity_rest_ratio"] = out["activity_class_1_ratio"]
    out["activity_inactive_ratio"] = out["activity_class_2_ratio"]
    out["activity_active_ratio"] = (
        out["activity_class_3_ratio"] + out["activity_class_4_ratio"] + out["activity_class_5_ratio"]
        if total
        else np.nan
    )
    out["activity_not_worn_ratio"] = out["activity_class_0_ratio"]
    out["activity_class_valid_count"] = total
    return out


def sleep_hypnogram_features(seq: np.ndarray) -> dict[str, float]:
    arr = np.asarray(seq, dtype=float)
    arr = arr[(arr != -1) & (arr != 0)]
    total = len(arr)
    out: dict[str, float] = {}
    stage_map = {1: "deep", 2: "light", 3: "rem", 4: "awake"}

    for level, name in stage_map.items():
        count = int(np.sum(arr == level))
        out[f"sleep_{name}_count_5min"] = count
        out[f"sleep_{name}_ratio_5min"] = count / total if total else np.nan

    out["sleep_stage_transition_count"] = int(np.sum(arr[1:] != arr[:-1])) if total > 1 else 0
    out["sleep_hypnogram_valid_count"] = total
    return out


def add_sequence_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    rows = []

    for _, row in df.iterrows():
        feats = {}
        feats.update(
            activity_class_features(
                parse_slash_sequence(row["CONVERT(activity_class_5min USING utf8)"], dtype=float)
            )
        )
        feats.update(
            numeric_sequence_stats(
                parse_slash_sequence(row["CONVERT(activity_met_1min USING utf8)"], dtype=float),
                "activity_met_1min",
                remove_zero=False,
            )
        )
        feats.update(
            numeric_sequence_stats(
                parse_slash_sequence(row["CONVERT(sleep_hr_5min USING utf8)"], dtype=float),
                "sleep_hr_5min",
                remove_zero=True,
            )
        )
        feats.update(
            numeric_sequence_stats(
                parse_slash_sequence(row["CONVERT(sleep_rmssd_5min USING utf8)"], dtype=float),
                "sleep_rmssd_5min",
                remove_zero=True,
            )
        )
        feats.update(
            sleep_hypnogram_features(
                parse_slash_sequence(row["CONVERT(sleep_hypnogram_5min USING utf8)"], dtype=float)
            )
        )
        rows.append(feats)

    return pd.concat([df.reset_index(drop=True), pd.DataFrame(rows)], axis=1)


def make_daily_table(activity: pd.DataFrame, sleep: pd.DataFrame, label: pd.DataFrame) -> pd.DataFrame:
    activity = coerce_numeric_columns(activity, keep={"EMAIL", "activity_day_start", "activity_day_end"})
    sleep = coerce_numeric_columns(sleep, keep={"EMAIL", "sleep_bedtime_start", "sleep_bedtime_end"})

    activity["activity_day_start_dt"] = pd.to_datetime(activity["activity_day_start"], errors="coerce")
    activity["activity_day_end_dt"] = pd.to_datetime(activity["activity_day_end"], errors="coerce")
    sleep["sleep_bedtime_start_dt"] = pd.to_datetime(sleep["sleep_bedtime_start"], errors="coerce")
    sleep["sleep_bedtime_end_dt"] = pd.to_datetime(sleep["sleep_bedtime_end"], errors="coerce")

    activity["date"] = activity["activity_day_start_dt"].dt.date
    sleep["date"] = sleep["sleep_bedtime_end_dt"].dt.date

    for col in ("activity_day_start_dt", "sleep_bedtime_start_dt", "sleep_bedtime_end_dt", "date"):
        source = activity if col.startswith("activity") else sleep
        if source[col].isna().any():
            raise ValueError(f"{col} has unparsable values.")

    sleep_start = sleep["sleep_bedtime_start_dt"]
    sleep_end = sleep["sleep_bedtime_end_dt"]
    sleep["sleep_start_hour"] = sleep_start.dt.hour + sleep_start.dt.minute / 60 + sleep_start.dt.second / 3600
    sleep["sleep_end_hour"] = sleep_end.dt.hour + sleep_end.dt.minute / 60 + sleep_end.dt.second / 3600
    sleep["sleep_time_calculated"] = (sleep_end - sleep_start).dt.total_seconds() / 3600
    sleep["_sleep_duration_seconds"] = (sleep_end - sleep_start).dt.total_seconds()

    sleep = (
        sleep.sort_values(["EMAIL", "date", "_sleep_duration_seconds"], ascending=[True, True, False])
        .drop_duplicates(["EMAIL", "date"], keep="first")
        .reset_index(drop=True)
    )

    if activity.duplicated(["EMAIL", "date"]).any():
        dup = activity.loc[activity.duplicated(["EMAIL", "date"], keep=False), ["EMAIL", "date"]].head()
        raise ValueError(f"Activity has duplicate EMAIL/date rows. Examples:\n{dup}")

    daily = activity.merge(sleep, on=["EMAIL", "date"], how="inner", suffixes=("", "_sleep"))
    daily = daily.merge(label, on="EMAIL", how="left")
    if daily["label"].isna().any():
        raise ValueError("Some merged rows have missing labels.")

    daily = add_sequence_features(daily)
    daily = drop_unmodelable_columns(daily)
    daily = drop_single_value_columns(daily)
    daily = daily.replace([np.inf, -np.inf], np.nan)
    return daily


def drop_unmodelable_columns(df: pd.DataFrame) -> pd.DataFrame:
    drop_cols = [
        "activity_class_5min",
        "activity_met_1min",
        "sleep_hr_5min",
        "sleep_hypnogram_5min",
        "sleep_rmssd_5min",
        "CONVERT(activity_class_5min USING utf8)",
        "CONVERT(activity_met_1min USING utf8)",
        "CONVERT(sleep_hr_5min USING utf8)",
        "CONVERT(sleep_hypnogram_5min USING utf8)",
        "CONVERT(sleep_rmssd_5min USING utf8)",
        "activity_day_start",
        "activity_day_end",
        "activity_day_start_dt",
        "activity_day_end_dt",
        "sleep_bedtime_start",
        "sleep_bedtime_end",
        "sleep_bedtime_start_dt",
        "sleep_bedtime_end_dt",
        "_sleep_duration_seconds",
    ]
    return df.drop(columns=drop_cols, errors="ignore")


def drop_single_value_columns(df: pd.DataFrame) -> pd.DataFrame:
    protected = {"EMAIL", "date", "DIAG_NM", "original_label", "label"}
    drop_cols = [
        col for col in df.columns
        if col not in protected and df[col].dropna().nunique() <= 1
    ]
    return df.drop(columns=drop_cols)


def add_folds(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["fold"] = -1
    splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for fold, (_, valid_idx) in enumerate(splitter.split(df, df["label"])):
        df.loc[valid_idx, "fold"] = fold
    return df


def load_inputs() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_activity = read_csv_flexible(find_csv("train_activity.csv"))
    val_activity = read_csv_flexible(find_csv("val_activity.csv"))
    train_sleep = read_csv_flexible(find_csv("train_sleep.csv"))
    val_sleep = read_csv_flexible(find_csv("val_sleep.csv"))

    train_label_name = "training_label.csv" if (DATA_DIR / "training_label.csv").exists() else "training_label_activity.csv"
    val_label_name = "val_label.csv" if (DATA_DIR / "val_label.csv").exists() else "val_label_activity.csv"
    train_label = preprocess_label(read_csv_flexible(find_csv(train_label_name)))
    val_label = preprocess_label(read_csv_flexible(find_csv(val_label_name)))

    return train_activity, train_sleep, train_label, val_activity, val_sleep, val_label

def make_patient_table(daily_df: pd.DataFrame) -> pd.DataFrame:
    numeric_cols = daily_df.select_dtypes(include=[np.number]).columns
    exclude = {"original_label", "label", "fold"}
    features_to_agg = [c for c in numeric_cols if c not in exclude]

    df_mean = daily_df.groupby("EMAIL")[features_to_agg].mean().reset_index()
    
    df_std = daily_df.groupby("EMAIL")[features_to_agg].std().reset_index()
    std_rename = {col: col + "_std" for col in features_to_agg}
    df_std = df_std.rename(columns=std_rename)
    
    df_combined = df_mean.merge(df_std, on="EMAIL", how="left")
    df_combined = df_combined.fillna(0) # std가 NaN인 경우(1일치 데이터) 0으로 처리
    
    labels = daily_df[["EMAIL", "DIAG_NM", "original_label", "label"]].drop_duplicates()
    return df_combined.merge(labels, on="EMAIL", how="left")


def main() -> None:
    print(f"BASE_DIR: {BASE_DIR}")
    print(f"DATA_DIR: {DATA_DIR}")
    print(f"OUT_DIR: {OUT_DIR}")
    print("[1/4] Loading CSV files...")
    train_activity, train_sleep, train_label, val_activity, val_sleep, val_label = load_inputs()

    print("[2/4] Applying paper-style preprocessing...")
    train_daily = make_daily_table(train_activity, train_sleep, train_label)
    val_daily = make_daily_table(val_activity, val_sleep, val_label)

    print("[3/4] Adding 5-fold split to train data...")
    train_with_fold = add_folds(train_daily)

    print("[4/4] Saving daily outputs...")
    train_daily.to_csv(OUT_DIR / "train_tabular_base.csv", index=False, encoding="utf-8-sig")
    val_daily.to_csv(OUT_DIR / "val_tabular_base.csv", index=False, encoding="utf-8-sig")
    train_with_fold.to_csv(OUT_DIR / "train_lgbm_rf_binary_base_with_fold.csv", index=False, encoding="utf-8-sig")
    val_daily.to_csv(OUT_DIR / "test_lgbm_rf_binary_base_holdout.csv", index=False, encoding="utf-8-sig")

    print("[5/5] Creating and saving patient-level aggregated data (V2: mean + std)...")
    all_daily = pd.concat([train_daily, val_daily], ignore_index=True)
    patient_df = make_patient_table(all_daily)
    patient_df.to_csv(OUT_DIR / "patient_level_all_v2.csv", index=False, encoding="utf-8-sig")

    print("\nDone.")
    print("train daily:", train_daily.shape, "subjects:", train_daily["EMAIL"].nunique())
    print("val/test daily:", val_daily.shape, "subjects:", val_daily["EMAIL"].nunique())
    print("patient_level_all:", patient_df.shape)
    print("patient_level_all label distribution:")
    print(patient_df["DIAG_NM"].value_counts())


if __name__ == "__main__":
    main()



In [ ]:
def load_data():
    if not PATIENT_PATH.exists():
        raise FileNotFoundError(f"데이터 파일이 존재하지 않습니다: {PATIENT_PATH}")
    
    df = pd.read_csv(PATIENT_PATH)
    all_feats = [c for c in df.columns if c not in DROP_COLS and pd.api.types.is_numeric_dtype(df[c])]
    df[all_feats] = df[all_feats].replace([np.inf, -np.inf], np.nan)
    return df.reset_index(drop=True), all_feats


# 3. SHAP 기반 Forward Selection

**📌 참고**: SMOTE를 활용한 핵심 전처리(오버샘플링)는 Data Leakage를 막기 위해 아래 교차검증(CV) 루프 내부에 구현되어 있습니다.


In [ ]:
def perform_shap_forward_selection(df, features):
    print("\n[단계 1] SHAP 패키지를 활용한 랭킹 계산 및 Forward Feature Selection 탐색...")
    X = df[features]
    y = df[TARGET_COL].astype(int)
    
    smote = SMOTE(random_state=RANDOM_STATE)
    X_res, y_res = smote.fit_resample(X, y)
    
    base_model = LGBMClassifier(random_state=RANDOM_STATE, n_jobs=1, class_weight='balanced', verbose=-1)
    base_model.fit(X_res, y_res)
    
    analyzer = ShapAnalyzer(model=base_model, feature_names=features, task="binary", n_classes=1, class_names=["Abnormal"])
    analyzer.explain(X_res)
    shap_df = analyzer.to_dataframe(combine_classes=False)
    
    ranked_features = shap_df['feature'].tolist()
    top_k_features = ranked_features[:FORWARD_SELECTION_MAX_FEATURES]
    
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    best_k = 0
    best_cv_score = -1
    history_scores = []
    
    for k in range(1, len(top_k_features) + 1):
        curr_feats = top_k_features[:k]
        X_sub = X[curr_feats]
        
        fold_scores = []
        for train_idx, val_idx in skf.split(X_sub, y):
            X_tr, y_tr = X_sub.iloc[train_idx], y.iloc[train_idx]
            X_va, y_va = X_sub.iloc[val_idx], y.iloc[val_idx]
            
            X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)
            
            eval_model = LGBMClassifier(
                random_state=RANDOM_STATE,
                n_jobs=1,
                num_leaves=33,
                learning_rate=0.08,
                n_estimators=120,
                min_child_samples=15,
                class_weight='balanced',
                verbose=-1
            )
            eval_model.fit(
                X_tr_res, y_tr_res,
                eval_set=[(X_va, y_va)],
                callbacks=[lgb.early_stopping(30, verbose=False)]
            )
            prob = eval_model.predict_proba(X_va)[:, 1]
            fold_scores.append(roc_auc_score(y_va, prob))
            
        mean_auc = np.mean(fold_scores)
        history_scores.append(mean_auc)
        
        if mean_auc > best_cv_score:
            best_cv_score = mean_auc
            best_k = k
            
        if k % 10 == 0 or k == len(top_k_features):
            print(f"  -> SHAP 상위 피처 {k:2d}개 적용 시 CV AUC: {mean_auc:.4f} (현재 최고: K={best_k}, AUC={best_cv_score:.4f})")
            
    optimal_features = top_k_features[:best_k]
    print(f"\n[SHAP 피처 탐색 완료] 최적 피처 개수: {best_k}개 (Best CV AUC: {best_cv_score:.4f})")
    return optimal_features, history_scores


# 4. SOTA 앙상블 모델 학습 및 Grid Search

**📌 참고**: SMOTE를 활용한 핵심 전처리(오버샘플링)는 Data Leakage를 막기 위해 아래 교차검증(CV) 루프 내부에 구현되어 있습니다.


In [ ]:
def run_sota_binary_ensemble(df, features):
    print("\n[단계 2] 4개 시그니처 알고리즘 (LightGBM, CatBoost, XGBoost, RF) 최적화 5-Fold 학습...")
    X = df[features]
    y = df[TARGET_COL].astype(int)
    
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    smote = SMOTE(random_state=RANDOM_STATE)
    
    models = ["LightGBM", "CatBoost", "XGBoost", "RandomForest", "Ensemble"]
    fold_predictions = {m: [] for m in models}
    fold_y_true = []
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_te, y_te = X.iloc[test_idx], y.iloc[test_idx]
        
        X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)
        
        # 1. LightGBM (V2 SOTA Hyperparameters)
        m_lgb = LGBMClassifier(
            objective="binary",
            num_leaves=33,
            learning_rate=0.08,
            n_estimators=1000,
            min_child_samples=41,
            max_depth=5,
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=1,
            verbose=-1
        )
        m_lgb.fit(X_tr_res, y_tr_res, eval_set=[(X_te, y_te)], callbacks=[lgb.early_stopping(50, verbose=False)])
        prob_lgb = m_lgb.predict_proba(X_te)[:, 1]
        
        # 2. CatBoost (Regulated)
        m_cat = CatBoostClassifier(
            random_state=RANDOM_STATE, thread_count=1,
            depth=5, learning_rate=0.05, iterations=500,
            l2_leaf_reg=4.0, auto_class_weights='Balanced', verbose=False
        )
        m_cat.fit(X_tr_res, y_tr_res, eval_set=(X_te, y_te), early_stopping_rounds=50)
        prob_cat = m_cat.predict_proba(X_te)[:, 1]
        
        # 3. XGBoost
        m_xgb = XGBClassifier(
            random_state=RANDOM_STATE, n_jobs=1,
            max_depth=4, learning_rate=0.05, n_estimators=400,
            subsample=0.8, colsample_bytree=0.8, reg_alpha=0.2, reg_lambda=1.5,
            eval_metric='auc', early_stopping_rounds=50
        )
        m_xgb.fit(X_tr_res, y_tr_res, eval_set=[(X_te, y_te)], verbose=False)
        prob_xgb = m_xgb.predict_proba(X_te)[:, 1]
        
        # 4. RandomForest (V2 SOTA Params)
        m_rf = RandomForestClassifier(
            random_state=RANDOM_STATE, n_jobs=1,
            max_depth=10, n_estimators=1000, class_weight='balanced'
        )
        m_rf.fit(X_tr_res, y_tr_res)
        prob_rf = m_rf.predict_proba(X_te)[:, 1]
        
        # 5. Soft Voting Ensemble (LightGBM 40% + CatBoost 20% + XGBoost 20% + RF 20%)
        prob_ens = (prob_lgb * 0.40 + prob_cat * 0.20 + prob_xgb * 0.20 + prob_rf * 0.20)
        
        fold_predictions["LightGBM"].extend(prob_lgb)
        fold_predictions["CatBoost"].extend(prob_cat)
        fold_predictions["XGBoost"].extend(prob_xgb)
        fold_predictions["RandomForest"].extend(prob_rf)
        fold_predictions["Ensemble"].extend(prob_ens)
        fold_y_true.extend(y_te)
        
    y_true_all = np.array(fold_y_true)
    results = {}
    
    print("\n" + "="*75)
    print(" V26 이진 분류 SOTA 성능 평가 결과 (5-Fold Out-of-Fold Cross Validation)")
    print("="*75)
    
    for m in models:
        probs = np.array(fold_predictions[m])
        auc = roc_auc_score(y_true_all, probs)
        
        # Youden's J index로 최적 임계값 탐색
        fpr, tpr, thresholds = roc_curve(y_true_all, probs)
        best_idx = np.argmax(tpr - fpr)
        opt_thresh = thresholds[best_idx]
        
        preds = np.where(probs >= opt_thresh, 1, 0)
        acc = accuracy_score(y_true_all, preds)
        prec = precision_score(y_true_all, preds, zero_division=0)
        rec = recall_score(y_true_all, preds, zero_division=0)
        f1 = f1_score(y_true_all, preds, zero_division=0)
        cm = confusion_matrix(y_true_all, preds)
        
        results[m] = {
            "auc": auc, "acc": acc, "prec": prec, "rec": rec, "f1": f1,
            "threshold": opt_thresh, "cm": cm, "probs": probs, "y_true": y_true_all
        }
        
        prefix = "[SOTA] " if auc >= 0.77 or acc >= 0.77 else "       "
        print(f"{prefix}[{m:12s}] Acc: {acc:.4f} | Prec: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | ROC-AUC: {auc:.4f} (Thresh: {opt_thresh:.3f})")
        
    return results


# 5. 시각화 그래프 생성


In [ ]:
def plot_results(history_scores, optimal_k, results):
    # 1. Forward Selection Curve
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, len(history_scores) + 1), history_scores, marker='o', color='#2b5c8f', linewidth=2)
    plt.axvline(x=optimal_k, color='#e74c3c', linestyle='--', linewidth=2, label=f'Optimal K = {optimal_k}')
    plt.title('V26 SHAP Forward Selection Curve (5-Fold CV AUC)', fontsize=14, fontweight='bold')
    plt.xlabel('Number of Selected Features', fontsize=12)
    plt.ylabel('Mean ROC-AUC Score', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "forward_selection_v26.png", dpi=150)
    plt.close()
    
    # 2. Confusion Matrix Heatmaps
    fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))
    class_names = ['Normal(CN)', 'Abnormal']
    
    for ax, (m, r) in zip(axes, results.items()):
        sns.heatmap(r['cm'], annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, annot_kws={"size": 14}, ax=ax)
        ax.set_title(f"[{m}]\nAUC: {r['auc']:.4f} | Acc: {r['acc']:.4f}", fontsize=11, fontweight='bold')
        ax.set_xlabel('Predicted Label')
        ax.set_ylabel('True Label')
        
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "confusion_matrix_v26_binary.png", dpi=150)
    plt.close()
    
    # 3. ROC Curves
    plt.figure(figsize=(9, 7))
    for m, r in results.items():
        fpr, tpr, _ = roc_curve(r['y_true'], r['probs'])
        lw = 3 if m == "Ensemble" else 1.5
        ls = '-' if m == "Ensemble" else '--'
        plt.plot(fpr, tpr, label=f"{m} (AUC = {r['auc']:.4f})", linewidth=lw, linestyle=ls)
        
    plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    plt.title('V26 Binary Classification ROC Curves Comparison', fontsize=14, fontweight='bold')
    plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
    plt.ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=12)
    plt.legend(fontsize=11, loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "roc_curves_v26_binary.png", dpi=150)
    plt.close()
    
    print(f"\n[시각화 저장 완료] 결과 그래프가 {PLOT_DIR} 에 성공적으로 저장되었습니다.")


# 6. 메인 실행


In [ ]:
if __name__ == "__main__":
    start_t = datetime.now()
    print("V26 Binary Classification Optimization Execution Started.")
    
    # 1. 로드
    df, raw_features = load_data()
    
    # 2. SHAP 피처 선택
    opt_features, forward_hist = perform_shap_forward_selection(df, raw_features)
    
    # 3. SOTA 앙상블 모델 학습 & 평가
    results = run_sota_binary_ensemble(df, opt_features)
    
    # 4. 결과 시각화
    plot_results(forward_hist, len(opt_features), results)
    
    elapsed = datetime.now() - start_t
    print(f"\nCompleted in {elapsed}")
